In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('/content/train.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
# Convert date columns
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True, errors='coerce')

In [ ]:
# Create temporal features
df['Order_Month'] = df['Order Date'].dt.month
df['Order_Year'] = df['Order Date'].dt.year
df['Ship_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

In [ ]:
# Replace negative Ship_Days with NaN and fill missing values with median
df.loc[df['Ship_Days'] < 0, 'Ship_Days'] = np.nan
df['Ship_Days'] = df['Ship_Days'].fillna(df['Ship_Days'].median())

In [ ]:
df.isna().sum()

In [ ]:
df.head()

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')

In [ ]:
# Distribution of Sales
plt.figure(figsize=(8,4))
df['Sales'].hist(bins=30)
plt.title('Distribution of Sales')
plt.xlabel('Sales')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Average Sales by Category
plt.figure(figsize=(10,4))
category_sales = df.groupby('Category')['Sales'].mean().sort_values()
category_sales.plot(kind='barh')
plt.title('Average Sales by Category')
plt.xlabel('Average Sales')
plt.ylabel('Category')
plt.show()

In [ ]:
# Average Sales by Region
plt.figure(figsize=(10,4))
sns.barplot(x='Region', y='Sales', data=df)
plt.title('Average Sales by Region')
plt.xlabel('Region')
plt.ylabel('Average Sales')
plt.show()

In [ ]:
# Discount vs Sales
plt.figure(figsize=(8,4))
sns.scatterplot(x='Discount', y='Sales', data=df)
plt.title('Discount vs Sales')
plt.xlabel('Discount')
plt.ylabel('Sales')
plt.show()

In [ ]:
# Quantity vs Sales
plt.figure(figsize=(8,4))
sns.scatterplot(x='Quantity', y='Sales', data=df)
plt.title('Quantity vs Sales')
plt.xlabel('Quantity')
plt.ylabel('Sales')
plt.show()

In [ ]:
# Drop irrelevant columns
drop_cols = [
    'Row ID', 'Order ID', 'Customer ID', 'Customer Name',
    'Product ID', 'Product Name', 'Order Date', 'Ship Date',
    'Country', 'City', 'State', 'Postal Code', 'Profit'
]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

In [ ]:
# Sales is the main regression target.
# High_Sales is only a classification target for Logistic Regression.
# 1 means Sales above median.
# 0 means Sales equal to or below median.
median_sales = df['Sales'].median()
df['High_Sales'] = (df['Sales'] > median_sales).astype(int)

In [ ]:
sns.countplot(x='Segment', hue='High_Sales', data=df)
plt.title('High Sales by Segment')
plt.xlabel('Segment')
plt.ylabel('Count')
plt.show()

In [ ]:
sns.countplot(x='Category', hue='High_Sales', data=df)
plt.title('High Sales by Category')
plt.xlabel('Category')
plt.ylabel('Count')
plt.show()

In [ ]:
# Linear Regression uses one-hot encoding (pd.get_dummies)
# because it treats numbers as ordered values,
# so LabelEncoder is inappropriate for unordered categories.
df_lr = df.copy()
y_reg = df_lr['Sales']
X_reg = df_lr.drop(columns=['Sales', 'High_Sales'])
X_reg = pd.get_dummies(X_reg, drop_first=True)

In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_reg, y_train_reg)
lr_pred = lr_model.predict(X_test_reg)

In [ ]:
print("Linear Regression R2 Score:", lr_model.score(X_test_reg, y_test_reg))

In [ ]:
mae = np.mean(abs(y_test_reg - lr_pred))
rmse = np.sqrt(np.mean((y_test_reg - lr_pred) ** 2))
print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
df_log = df.copy()

categorical_cols = df_log.select_dtypes(include=['object']).columns

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_log[col] = le.fit_transform(df_log[col].astype(str))
    label_encoders[col] = le

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(10,4))
    sns.barplot(x=col, y='High_Sales', data=df_log)
    plt.title('High Sales Rate by ' + col)
    plt.xlabel(col)
    plt.ylabel('Average High_Sales')
    plt.show()

In [ ]:
X_le = df_log.drop(columns=['Sales', 'High_Sales'])
y_clf_le = df_log['High_Sales']

In [ ]:
X_train_le, X_test_le, y_train_le, y_test_le = train_test_split(
    X_le, y_clf_le, test_size=0.2, random_state=42
)

In [ ]:
log_model_le = LogisticRegression(max_iter=1000)
log_model_le.fit(X_train_le, y_train_le)
log_pred_le = log_model_le.predict(X_test_le)

In [ ]:
print("Logistic Regression Accuracy:", log_model_le.score(X_test_le, y_test_le))

In [ ]:
manual_accuracy = np.mean(y_test_le.values == log_pred_le)
print("Manual Accuracy:", manual_accuracy)

In [ ]:
pd.crosstab(
    y_test_le,
    log_pred_le,
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
combined_prediction = pd.DataFrame({
    'Actual Sales': y_test_reg.values,
    'Predicted Sales': lr_pred,
    'Actual High_Sales': y_test_le.values,
    'Predicted High_Sales': log_pred_le
})

combined_prediction.head(10)